"""
Ingestão do relatório de segurança pública -> vetores, com tratamento
DEDICADO para tabelas.

Por que mudar em relação ao PyPDFDirectoryLoader "puro":
- PyPDFDirectoryLoader extrai só o texto corrido da página. Tabelas viram uma
  sequência de números/palavras sem separação de coluna -> o embedding não
  consegue associar "12.345" à coluna "Roubos" e à linha "2023".
- Aqui usamos pdfplumber por página:
    1. Extrai as tabelas separadamente e converte cada uma em Markdown
       (mantém cabeçalho de coluna + linha, vira um Document com
       metadata tipo="tabela").
    2. O texto restante da página (fora das tabelas) segue pro
       RecursiveCharacterTextSplitter normal, igual ao pipeline antigo
       (metadata tipo="texto").
    3. Tabela NUNCA é cortada pelo splitter de tamanho fixo — ou fica
       inteira num chunk, ou (se for muito grande) é dividida repetindo
       o cabeçalho em cada pedaço, pra nunca perder o significado das
       colunas.

Requisito adicional: pip install pdfplumber
"""

In [26]:
# import os
# from dotenv import load_dotenv
# import warnings
# warnings.filterwarnings('ignore')
# import datetime
# import time
# import requests
# from tqdm.auto import tqdm



# # # Modelos LLM (Large Language Models)
# from langchain_google_genai import ChatGoogleGenerativeAI
# from langchain_core.language_models.chat_models import BaseChatModel

# #embedding
# from langchain_huggingface import HuggingFaceEmbeddings  



# from langchain_core.prompts import (
#     PromptTemplate,
#     ChatPromptTemplate,
#     MessagesPlaceholder,
#     SystemMessagePromptTemplate,
#     HumanMessagePromptTemplate
# )

# from langchain_core.output_parsers import StrOutputParser


# # Criação e execução de agentes
# from langchain_classic.agents import( 
# Tool, 
# AgentExecutor,
# create_tool_calling_agent,
# create_react_agent)

# # # Ferramentas customizadas para agentes
# from langchain.tools import tool
# from langchain_community.agent_toolkits.load_tools import load_tools
# from langchain_experimental.tools.python.tool import PythonAstREPLTool

# from langchain_classic.memory import ConversationBufferMemory


# from langchain_google_genai import GoogleGenerativeAIEmbeddings
# from langchain_community.document_loaders import PyPDFDirectoryLoader
# from langchain_text_splitters import RecursiveCharacterTextSplitter,MarkdownHeaderTextSplitter

# # # Componentes de RAG (Retrieval-Augmented Generation)
# from langchain_chroma import Chroma  # Armazenamento vetorial


# print('#',time.strftime("%d/%m/%Y - %H:%M:%S"))



# 12/08/2026 - 15:36:55


In [1]:
import os

import re
import time
from pathlib import Path
from dotenv import load_dotenv

import pdfplumber
from tqdm import tqdm
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

import warnings
warnings.filterwarnings('ignore')

OUTPUT_DOCUMENTS_DIR = Path("./fonte/")
VECTORSTORE_DIR = Path("./vectorstore_teste_large")



ENV_PATH: str = '/home/akel/PycharmProjects/InsurMinds2026/.env'

def carrega_variaveis_ambiente() -> None:
    if os.path.exists(ENV_PATH):
        load_dotenv(ENV_PATH, override=True)
        print("✔ Variáveis de ambiente carregadas do arquivo .env")
    else:
        print(f"⚠ Aviso: Arquivo {ENV_PATH} não foi encontrado no diretório atual.")

print('✔ OUTPUT_DOCUMENTS_DIR:',OUTPUT_DOCUMENTS_DIR)
carrega_variaveis_ambiente()
print('#',time.strftime("%d/%m/%Y - %H:%M:%S"))



✔ OUTPUT_DOCUMENTS_DIR: fonte
✔ Variáveis de ambiente carregadas do arquivo .env
# 12/08/2026 - 17:02:26


## imports

## Funções

In [2]:
TITULO_REGEX = re.compile(
    r"^(tabela|quadro|gr[aá]fico|figura)\s*[\d\.]+",
    re.IGNORECASE
)

def _tabela_para_markdown(dados: list[list]) -> str:
    """Converte lista-de-listas do pdfplumber em Markdown com cabeçalho."""
    linhas = [
        [(c or "").strip().replace("\n", " ") for c in row]
        for row in dados if any(c for c in row)
    ]
    if len(linhas) < 2:
        return ""
    cab = linhas[0]
    md  = "| " + " | ".join(cab) + " |\n"
    md += "|" + "|".join(["---"] * len(cab)) + "|\n"
    for row in linhas[1:]:
        row_adj = (row + [""] * len(cab))[:len(cab)]
        md += "| " + " | ".join(row_adj) + " |\n"
    return md

def _titulo_proximo(linhas_pagina: list[str]) -> str:
    """Retorna a legenda/título mais próximo antes de uma tabela."""
    for linha in reversed(linhas_pagina[-6:]):
        if TITULO_REGEX.search(linha):
            return linha.strip()
    return ""

def carregar_com_pdfplumber(diretorio: str | Path) -> tuple[list[Document], list[Document]]:
    """
    Retorna:
        docs_texto  -> páginas de texto narrativo (sem a área das tabelas*)
        docs_tabela -> uma tabela por Document, em Markdown

    * pdfplumber não remove automaticamente o texto da área da tabela,
      então ele aparece nos dois. Isso é aceitável: o chunk de tabela tem
      estrutura de coluna; o chunk de texto tem contexto narrativo.
    """
    docs_texto, docs_tabela = [], []
    pdfs = sorted(Path(diretorio).glob("*.pdf"))
    print(f"✔ {len(pdfs)} PDF(s) encontrado(s)")

    for pdf_path in pdfs:
        with pdfplumber.open(pdf_path) as pdf:
            for num_pag, pag in enumerate(pdf.pages):
                texto = pag.extract_text() or ""
                linhas = texto.split("\n")

                # — tabelas —
                for idx_tab, tbl in enumerate(pag.find_tables()):
                    md = _tabela_para_markdown(tbl.extract())
                    if not md:
                        continue
                    docs_tabela.append(Document(
                        page_content=md,
                        metadata={
                            "source"           : str(pdf_path),
                            "page"             : num_pag,          # mantém a chave "page" do PyPDF
                            "tipo"             : "tabela",
                            "titulo_tabela"    : _titulo_proximo(linhas),
                            "tabela_idx_pagina": idx_tab,
                        }
                    ))

                # — texto narrativo —
                if texto.strip():
                    docs_texto.append(Document(
                        page_content=texto,
                        metadata={
                            "source": str(pdf_path),
                            "page"  : num_pag,
                            "tipo"  : "texto",
                        }
                    ))

        n_tab = sum(1 for d in docs_tabela if d.metadata["source"] == str(pdf_path))
        n_txt = sum(1 for d in docs_texto  if d.metadata["source"] == str(pdf_path))
        print(f"  {pdf_path.name}: {n_txt} páginas de texto | {n_tab} tabelas")

    return docs_texto, docs_tabela
# ==========================================
# 1. EXTRAÇÃO — texto e tabelas separados
# ==========================================
def extrair_titulo_tabela(linhas_antes: list[str]) -> str:
    """Procura, nas últimas linhas antes da tabela, algo como 'Tabela 3 - Ocorrências...'."""
    for linha in reversed(linhas_antes[-5:]):
        if TITULO_TABELA_REGEX.search(linha):
            return linha.strip()
    return ""


def tabela_para_markdown(tabela: list[list]) -> str:
    """Converte uma tabela extraída pelo pdfplumber (lista de linhas) em Markdown."""
    linhas_limpas = [
        [(cel or "").strip().replace("\n", " ") for cel in linha]
        for linha in tabela
        if any(cel for cel in linha)
    ]
    if len(linhas_limpas) < 2:
        return ""

    cabecalho = linhas_limpas[0]
    corpo = linhas_limpas[1:]

    md = "| " + " | ".join(cabecalho) + " |\n"
    md += "|" + "|".join(["---"] * len(cabecalho)) + "|\n"
    for linha in corpo:
        linha_ajustada = (linha + [""] * len(cabecalho))[:len(cabecalho)]
        md += "| " + " | ".join(linha_ajustada) + " |\n"

    return md


def processar_pdf(caminho_pdf: Path) -> tuple[list[Document], list[Document]]:
    """Retorna (documentos_texto, documentos_tabela) de um único PDF."""
    docs_texto, docs_tabela = [], []

    with pdfplumber.open(caminho_pdf) as pdf:
        for num_pagina, pagina in enumerate(pdf.pages):
            texto_completo = pagina.extract_text() or ""
            linhas = texto_completo.split("\n")

            tabelas = pagina.find_tables()
            for idx_tab, tabela in enumerate(tabelas):
                dados = tabela.extract()
                md_tabela = tabela_para_markdown(dados)
                if not md_tabela:
                    continue

                titulo = extrair_titulo_tabela(linhas)

                docs_tabela.append(Document(
                    page_content=md_tabela,
                    metadata={
                        "source": str(caminho_pdf),
                        "page": num_pagina,
                        "tipo": "tabela",
                        "titulo_tabela": titulo or f"Tabela sem título (pág. {num_pagina + 1})",
                        "tabela_idx_na_pagina": idx_tab,
                    }
                ))

            # Nota: o texto da página continua incluindo os números da tabela
            # (o pdfplumber não remove automaticamente). Isso é aceitável:
            # o BM25/vetor pode até achar esse texto, mas a resposta "boa"
            # vem do chunk tipo="tabela", que tem estrutura de coluna.
            if texto_completo.strip():
                docs_texto.append(Document(
                    page_content=texto_completo,
                    metadata={
                        "source": str(caminho_pdf),
                        "page": num_pagina,
                        "tipo": "texto",
                    }
                ))

    return docs_texto, docs_tabela


def carregar_documentos(diretorio: Path) -> tuple[list[Document], list[Document]]:
    todos_texto, todas_tabelas = [], []
    pdfs = sorted(diretorio.glob("*.pdf"))
    print(f"✔ {len(pdfs)} PDF(s) encontrado(s) em {diretorio}")

    for pdf_path in pdfs:
        docs_texto, docs_tabela = processar_pdf(pdf_path)
        todos_texto.extend(docs_texto)
        todas_tabelas.extend(docs_tabela)
        print(f"  - {pdf_path.name}: {len(docs_texto)} páginas de texto | {len(docs_tabela)} tabelas")

    return todos_texto, todas_tabelas


# ==========================================
# 2. SPLIT — só o texto narrativo é fatiado
# ==========================================
def split_tabela_grande(doc: Document, limite: int = 3000) -> list[Document]:
    """Tabelas > limite de caracteres são divididas repetindo o cabeçalho em cada parte."""
    linhas = doc.page_content.split("\n")
    if len(doc.page_content) <= limite or len(linhas) < 4:
        return [doc]

    cabecalho = linhas[0] + "\n" + linhas[1]  # linha de colunas + linha separadora "---"
    corpo = linhas[2:]

    partes, bloco_atual = [], []
    tamanho_atual = len(cabecalho)

    for linha in corpo:
        if tamanho_atual + len(linha) > limite and bloco_atual:
            conteudo = cabecalho + "\n" + "\n".join(bloco_atual)
            partes.append(Document(page_content=conteudo, metadata=dict(doc.metadata)))
            bloco_atual, tamanho_atual = [], len(cabecalho)
        bloco_atual.append(linha)
        tamanho_atual += len(linha)

    if bloco_atual:
        conteudo = cabecalho + "\n" + "\n".join(bloco_atual)
        partes.append(Document(page_content=conteudo, metadata=dict(doc.metadata)))

    for i, parte in enumerate(partes):
        parte.metadata["tabela_parte"] = f"{i + 1}/{len(partes)}"

    return partes
print('#',time.strftime("%d/%m/%Y - %H:%M:%S"))


# 12/08/2026 - 17:02:36


## EXECUCAO
### Carregar documento

In [3]:
# ==========================================
# EXECUÇÃO
# ==========================================

docs_texto, docs_tabela = carregar_com_pdfplumber(OUTPUT_DOCUMENTS_DIR)
print(f"\n✔ Carregamento concluído")
print(f"  Páginas de texto : {len(docs_texto)}")
print(f"  Tabelas extraídas: {len(docs_tabela)}")
print(f"# {time.strftime('%d/%m/%Y - %H:%M:%S')}")


✔ 1 PDF(s) encontrado(s)
  anuario_2026.pdf: 424 páginas de texto | 346 tabelas

✔ Carregamento concluído
  Páginas de texto : 424
  Tabelas extraídas: 346
# 12/08/2026 - 17:03:43


### Split

In [4]:
## PATCH 2 — substitui o bloco do RecursiveCharacterTextSplitter
##
## Regra nova:
##   • texto narrativo -> splitter normal (chunk_size 1000, overlap 150)
##   • tabela          -> NUNCA cortada pelo tamanho; se for muito grande,
##                        divide repetindo o cabeçalho em cada parte

CHUNK_SIZE    = 1000   # reduzido de 1200: texto técnico denso é mais preciso menor
CHUNK_OVERLAP = 150
MAX_TABLE_CHARS = 3000  # tabelas maiores que isso são divididas com repetição de cabeçalho

# ── 2a. Splitter para texto narrativo ────────────────────────────────────────
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ".", "!", "?", " "],
    length_function=len,
)
split_texto = text_splitter.split_documents(docs_texto)
print(f"✔ Texto fatiado: {len(split_texto)} chunks")


# ── 2b. Splitter para tabelas (nunca corta no meio) ──────────────────────────
def _split_tabela_grande(doc: Document, limite: int = MAX_TABLE_CHARS) -> list[Document]:
    """
    Se a tabela cabe no limite -> retorna como está (1 chunk).
    Se não cabe -> divide pelas linhas de dados, REPETINDO o cabeçalho
    em cada parte para que o embedding nunca veja uma linha sem contexto
    de coluna.
    """
    if len(doc.page_content) <= limite:
        return [doc]

    linhas = doc.page_content.split("\n")
    if len(linhas) < 4:          # tabela minúscula mesmo
        return [doc]

    cabecalho  = linhas[0] + "\n" + linhas[1]   # | col1 | col2 | ... e |---|---|...
    linhas_dados = linhas[2:]

    partes, bloco, tamanho = [], [], len(cabecalho)
    for linha in linhas_dados:
        if tamanho + len(linha) > limite and bloco:
            conteudo = cabecalho + "\n" + "\n".join(bloco)
            partes.append(Document(page_content=conteudo, metadata=dict(doc.metadata)))
            bloco, tamanho = [], len(cabecalho)
        bloco.append(linha)
        tamanho += len(linha)

    if bloco:
        conteudo = cabecalho + "\n" + "\n".join(bloco)
        partes.append(Document(page_content=conteudo, metadata=dict(doc.metadata)))

    for i, parte in enumerate(partes):
        parte.metadata["tabela_parte"] = f"{i + 1}/{len(partes)}"

    return partes


split_tabelas = []
for doc in docs_tabela:
    split_tabelas.extend(_split_tabela_grande(doc))
print(f"✔ Tabelas processadas: {len(split_tabelas)} chunks")


# ── 2c. Une tudo e numera ─────────────────────────────────────────────────────
# Tabelas vão DEPOIS do texto para que chunk_id 0..N_texto sejam o narrativo
split_documents = split_texto + split_tabelas

for i, split in enumerate(split_documents):
    split.metadata.update({
        "chunk_id"   : i,
        "chunk_total": len(split_documents),
        "posicao"    : f"{i / len(split_documents) * 100:.0f}%",
        # "tipo" já foi definido na extração ("texto" ou "tabela")
        # garantia de que nenhum chunk fique sem o campo:
        "tipo"       : split.metadata.get("tipo", "texto"),
    })

print(f"\n# Documento fatiado em: {time.strftime('%d/%m/%Y - %H:%M:%S')}")
print(f"  chunk_total (texto)  : {len(split_texto)}")
print(f"  chunk_total (tabelas): {len(split_tabelas)}")
print(f"  chunk_total (geral)  : {len(split_documents)}")
print("--------------------")

✔ Texto fatiado: 1327 chunks
✔ Tabelas processadas: 349 chunks

# Documento fatiado em: 12/08/2026 - 17:05:02
  chunk_total (texto)  : 1327
  chunk_total (tabelas): 349
  chunk_total (geral)  : 1676
--------------------


In [9]:
# text_splitter = RecursiveCharacterTextSplitter(
#     chunk_size=1200,        # mantido igual ao pipeline original
#     chunk_overlap=150,
#     separators=["\n\n", "\n", ".", "!", "?", " "],
#     length_function=len,
# )
# split_texto = text_splitter.split_documents(documentos_texto)

# split_tabelas = []
# for doc in documentos_tabela:
#     split_tabelas.extend(split_tabela_grande(doc))

# split_documents = split_texto + split_tabelas

# for i, split in enumerate(split_documents):
#     split.metadata.update({
#         "chunk_id": i,
#         "chunk_total": len(split_documents),
#         "posicao": f"{i / len(split_documents) * 100:.0f}%",
#     })

# print("# Documento fatiado em:", time.strftime("%d/%m/%Y - %H:%M:%S"))
# print(" chunk_total (texto)  :", len(split_texto))
# print(" chunk_total (tabelas):", len(split_tabelas))
# print(" chunk_total (geral)  :", len(split_documents))
# print("--------------------")



### Embedding

In [5]:
# ==========================================
# 3. EMBEDDING E INDEXAÇÃO (igual ao pipeline original)
# ==========================================
#embedding
model_embedding1='intfloat/multilingual-e5-small'
model_embedding2='intfloat/multilingual-e5-large-instruct'
embedding_e5 = HuggingFaceEmbeddings(
    model_name=model_embedding2,
    model_kwargs={"device": "cpu", "trust_remote_code": True},
    encode_kwargs={"normalize_embeddings": True, "prompt": "passage: "}
)

batch_size = 150
total = len(split_documents)

print(f"\n#Iniciando criação do banco: {total} chunks")
print(f"#Finalizado em: {time.strftime('%H:%M:%S')}")


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


#Iniciando criação do banco: 1676 chunks
#Finalizado em: 17:06:35


### vectorstore building

In [6]:
vectorstore = Chroma.from_documents(
    documents=split_documents[:batch_size],
    embedding=embedding_e5,
    persist_directory=str(VECTORSTORE_DIR),
    collection_metadata={"hnsw:space": "cosine"}
)

for i in tqdm(range(batch_size, total, batch_size), desc="Indexando chunks"):
    lote = split_documents[i:i + batch_size]
    vectorstore.add_documents(lote)
    print(f"  Lote {i // batch_size + 1}/{(total // batch_size) + 1} | "
          f"Chunks {i}-{min(i + batch_size, total)} | "
          f"{time.strftime('%H:%M:%S')}")

print(f"\n#Banco criado com {vectorstore._collection.count()} chunks.")
print(f"#Finalizado em: {time.strftime('%H:%M:%S')}")

Indexando chunks:   9%|█▉                   | 1/11 [22:53<3:48:52, 1373.28s/it]

  Lote 2/12 | Chunks 150-300 | 17:58:20


Indexando chunks:  18%|███▊                 | 2/11 [47:08<3:33:14, 1421.60s/it]

  Lote 3/12 | Chunks 300-450 | 18:22:35


Indexando chunks:  27%|█████▏             | 3/11 [1:12:18<3:14:56, 1462.10s/it]

...

  Lote 10/12 | Chunks 1350-1500 | 21:18:10


Indexando chunks:  91%|██████████████████▏ | 10/11 [4:06:43<24:49, 1489.70s/it]

  Lote 11/12 | Chunks 1500-1650 | 21:42:10


Indexando chunks: 100%|████████████████████| 11/11 [4:12:32<00:00, 1377.48s/it]

  Lote 12/12 | Chunks 1650-1676 | 21:47:59

#Banco criado com 1676 chunks.
#Finalizado em: 21:47:59


## leitura do banco de dados

In [7]:
def carrega_banco_de_dados_vetorial(path_documentos:str) -> Chroma:
    try:
        model_embedding1='intfloat/multilingual-e5-small'
        model_embedding2='intfloat/multilingual-e5-large-instruct'
        embedding_query = HuggingFaceEmbeddings(
            model_name=model_embedding2,
            model_kwargs={"device": "cpu",
                          "trust_remote_code": True},
            encode_kwargs={
                "normalize_embeddings": True,
                "prompt": "query: "   })

        vectorstore = Chroma(persist_directory=path_documentos, embedding_function=embedding_query)
        if vectorstore:
            print('banco de dados carregado!')
        
        return vectorstore
    except Exception as e:
        print(f"Erro ao carregar o banco de dados vetorial: {e}")
        return None

In [23]:
vectorstore = carrega_banco_de_dados_vetorial(f'{VECTORSTORE_DIR}')


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

banco de dados carregado!


In [24]:
if vectorstore:
    # Obtém dados armazenados (por padrão traz documentos, metadados e IDs)
    dados = vectorstore.get(
        limit=5, # Limite os resultados para não sobrecarregar a tela
        include=['metadatas', 'documents'] # Pode incluir também 'embeddings' se necessário
    )
    
    print("--- METADADOS DOS PRIMEIRAMENTE CARREGADOS ---")
    for i, (doc, meta) in enumerate(zip(dados['documents'], dados['metadatas'])):
        print(f"\nItem {i+1}:")
        print(f"Texto (primeiros 100 caracteres): {doc[:100]}...")
        print(f"Metadados: {meta}")

--- METADADOS DOS PRIMEIRAMENTE CARREGADOS ---

Item 1:
Texto (primeiros 100 caracteres): Anuário 
Brasileiro 
de Segurança 
Pública
2026...
Metadados: {'chunk_id': 0, 'source': 'fonte/anuario_2026.pdf', 'chunk_total': 1472, 'posicao': '0%', 'page': 0, 'tipo': 'texto'}

Item 2:
Texto (primeiros 100 caracteres): Anuário 
Brasileiro 
de Segurança 
Pública
2026...
Metadados: {'source': 'fonte/anuario_2026.pdf', 'page': 1, 'posicao': '0%', 'chunk_id': 1, 'chunk_total': 1472, 'tipo': 'texto'}

Item 3:
Texto (primeiros 100 caracteres): Anuário 
Brasileiro 
de Segurança 
Pública
2026
Ano 20 - 2026 
ISSN 1983-7364...
Metadados: {'posicao': '0%', 'source': 'fonte/anuario_2026.pdf', 'tipo': 'texto', 'chunk_id': 2, 'chunk_total': 1472, 'page': 2}

Item 4:
Texto (primeiros 100 caracteres): FICHA INSTITUCIONAL
Diretor Presidente Equipe Técnica Conselho de Administração
Renato Sérgio de Lim...
Metadados: {'posicao': '0%', 'page': 3, 'chunk_id': 3, 'source': 'fonte/anuario_2026.pdf', 'tipo': 'texto',

In [25]:
def busca_na_base_de_documentos(pergunta:str) -> str:
    """Use esta ferramenta para responder perguntas sobre o anuario de segurança publica."""
    vectorstore = carrega_banco_de_dados_vetorial(f'{VECTORSTORE_DIR}')
    contexto = None
    if vectorstore:
        retriever = vectorstore.as_retriever()
        docs = retriever.invoke(pergunta)
        contexto = "\n\n".join([doc.page_content for doc in docs])
    return contexto

In [28]:
llm_gemini = ChatGoogleGenerativeAI(temperature=0, model="gemini-3.1-flash-lite-preview",google_api_key=os.getenv("GOOGLE_API"))

def string_gemini(out_agent_exe):
    if isinstance(out_agent_exe, list) and len(out_agent_exe) > 0:
        if isinstance(out_agent_exe[0], dict) and 'text' in out_agent_exe[0]:
            return out_agent_exe[0]['text']
    
    return  out_agent_exe[0]['text']


def agente_langchain_RAG1(modelo_llm=llm_gemini) -> dict:
    ferramentas = []
    memoria = ConversationBufferMemory(memory_key="chat_history", return_messages=True, input_key="input")

    prompt = PromptTemplate(
        input_variables=["input", "context", "chat_history", "agent_scratchpad"],
        template=""" {chat_history}
                Você é um agente de IA especializado em responder perguntas de segurança publica. Responda apenas informações que estão 
                dentro do seu contexto.Jamais busque informações de outras fontes
                Contexto: {context}
                Pergunta: {input}
                {agent_scratchpad}
        """)

    agente = create_tool_calling_agent(modelo_llm,ferramentas, prompt)
    executor_do_agente = AgentExecutor(agent=agente, tools=ferramentas, memory=memoria)
    return executor_do_agente

In [29]:
executor_do_agente = agente_langchain_RAG1()

pergunta1 = "comente sobre taxa de Mortes Violentas Intencionais 2024 e 2025"
contexto1 = ''
resposta1 = executor_do_agente.invoke({"input": pergunta1, "context": contexto1})
resposta1=string_gemini(resposta1['output'])

print(resposta1)
print('\n' + '=' * 10)

/tmp/ipykernel_27565/2826231279.py:13: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memoria = ConversationBufferMemory(memory_key="chat_history", return_messages=True, input_key="input")


O contexto fornecido está vazio, portanto, não há informações disponíveis para responder à sua pergunta sobre as taxas de Mortes Violentas Intencionais de 2024 e 2025.



In [30]:
executor_do_agente = agente_langchain_RAG1()

pergunta1 = "Comente Roubos e Furtos de celular nas capitais.Números e evolução."
contexto = busca_na_base_de_documentos(pergunta1)
resposta2 = executor_do_agente.invoke({"input": pergunta1, "context": contexto})
resposta2=string_gemini(resposta2['output'])
#
print(resposta2)
print('\n' + '=' * 10)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

banco de dados carregado!
Com base no contexto fornecido do Anuário Brasileiro de Segurança Pública - 2026, seguem as informações sobre roubos e furtos de celulares:

**Dados de Taxas em Capitais e Municípios Selecionados:**
O anuário apresenta uma lista de municípios com suas respectivas taxas de roubo/furto de celular. Entre os dados disponíveis, destacam-se:

*   **Belém (PA):** 1.487,0
*   **Salvador (BA):** 1.195,0
*   **Porto Velho (RO):** 1.123,2
*   **Olinda (PE):** 1.060,3
*   **Teresina (PI):** 975,3
*   **Cariacica (ES):** 932,2
*   **Marituba (PA):** 863,2
*   **Belo Horizonte (MG):** 831,3
*   **Itapecerica da Serra (SP):** 801,2
*   **Rio de Janeiro (RJ):** 793,7

**Dinâmica e Evolução:**
*   **Análise Geral:** O documento destaca, no "Texto 05" (página 104), que os roubos e furtos de celular no Brasil possuem dinâmicas distintas.
*   **Evolução Temporal:** O "Gráfico 23" (página 105) apresenta o histórico das ocorrências de roubo e furto de celulares no Brasil no período